In [22]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import requests

In [23]:

from langchain.agents import create_agent


In [24]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [25]:
search_tool = TavilySearchResults(max_results=2)

In [26]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [27]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'title': 'Guides: Artificial Intelligence (Generative) Resources: Latest News on AI',
  'url': 'https://guides.library.georgetown.edu/ai/news',
  'content': "Privatized air traffic control would mean safer skies This link opens in a new window_by The Washington Post 20 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 \n   The U.S. and China need an arms control pact — for AI This link opens in a new window_by Asha Jadeja The Washington Post 20 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 \n   Elon Musk Beats Anthropic on Giveaways; He says he'll give every child a share of SpaceX, aka a stake in capitalism.This link opens in a new window_Wall Street Journal (Online) 19 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 [...] DHS Orders DeepSeek Technology Removed From Federal Civilian IT Systems; The order also bars future purchases of the company's products an

In [28]:

# Initialize the model (it automatically picks up GEMINI_API_KEY from your environment)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.7
)

# Invoke the model with a message
messages = [HumanMessage(content="What is capital of Delhi?")]
response = llm.invoke(messages)

print(response.content)

c:\Users\Satish\.conda\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Delhi is itself a union territory and a city, and it serves as the **national capital of India**. \n\nWithin Delhi, there is a specific district called **New Delhi**, which is the actual seat of the Government of India and functions as the capital city.', 'extras': {'signature': 'EmAKXgFpFH0TUTW9oH1jkD+m2YIAjzISvj+HupqELROpxhg1Et+TBLdjuySaZ6dgKAbHr9stRO4My0pA712cB8FfVdR3P521t+jPuHhTBLSEcB8opxYjaOlIXk2rbAs2z7Y='}}]


In [29]:
tools = [search_tool, get_weather_data]

In [30]:
from langchain.agents import create_agent

In [32]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "Answer in no more than 3 concise sentences."
    
    ),
    debug=False
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Find the  Latest new on France."
                "AND provide the current weather in Paris."
            )
        }
    ]
})

print(response["messages"][-1].content)

c:\Users\Satish\.conda\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


c:\Users\Satish\.conda\envs\langagent\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'France has recently made headlines for planning a record amount of borrowing on the bond market ahead of its upcoming budget announcement. Meanwhile, the current weather in Paris is sunny with a temperature of 14°C and 53% humidity.', 'extras': {'signature': 'EmAKXgFpFH0TdTU0TLsQ7v6CTCb6ESXlgctOTmfASVXXeqyq7pg6fYyXXA0GxtByOFHku6xvUpCeSFbVE/ZKXjISPk5TctPOVJg8F4ILnjgQ29w34FWOD483WQxcwPGKI0A='}}]
